# 2 What did that capacity buy you?

Notebook 01 measured one architecture family against a capacity-matched control, and the
control kept winning. This notebook asks the general version of that question.

You can spend resources in three places. More **parameters**, more **data**, more
**compute**. Each is a knob you can turn, each costs money, and the folklore says each
one buys you accuracy. So:

> Turn each knob over two orders of magnitude and **draw the curve**. Where does the
> metric actually move, and where are you paying for nothing?

This is what the scaling-laws literature does at a cost of millions of dollars per point.
On a cached representation a point costs seconds, so you can do it yourself — and the
answers on *your* data are not the answers in the papers.

Read alongside `references/Scaling_Laws_for_Neural_Language_Models(2020).pdf`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from notebooktester import param
from torch import nn
from vectormesh import VectorCache
from vectormesh.components import NeuralNet

SEEDS = param((0, 1, 2), test=(0,))
BATCH, LR = 128, 1e-3

cache = VectorCache.from_hub("pttrn-io/dtd-dinov2-small")

# Shuffle BEFORE slicing. The DTD caches are stored grouped by label, so `select(range(n))`
# would hand you a subset containing only the first few classes -- and the resulting
# "learning curve" would be a plot of how many classes you had, not of how much data helps.
d = cache.dataset.shuffle(seed=0).with_format("numpy")

X = torch.tensor(np.asarray(d["embed"], dtype="float32"))
y = torch.tensor(np.asarray(d["label"]))
DIM, N_CLASSES = X.shape[1], int(y.max()) + 1
ntr = int(0.75 * len(X))
Xtr, ytr, Xte, yte = X[:ntr], y[:ntr], X[ntr:], y[ntr:]

print(f"dim {DIM}   classes {N_CLASSES}   train {len(Xtr)}   test {len(Xte)}")

DTD — describable textures — is the lesson-1 dataset with real headroom. Flowers is
saturated at 0.99 and EuroSAT sits at 0.95 with almost no spread between encoders, so
neither would show a curve moving. DTD lands near 0.78, which leaves room for something
to happen.

In [ ]:
def accuracy(model, Xa, ya):
    model.eval()
    with torch.no_grad():
        return float((model(Xa).argmax(1) == ya).float().mean())


def n_params(model):
    return sum(p.numel() for p in model.parameters())


def fit(model, Xa, ya, epochs, checkpoints=None):
    """Train, optionally recording (train, test) accuracy at given epochs."""
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    lossf = nn.CrossEntropyLoss()
    seen = {}
    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(len(Xa))
        for i in range(0, len(Xa) - 1, BATCH):
            idx = perm[i:i + BATCH]
            opt.zero_grad()
            lossf(model(Xa[idx]), ya[idx]).backward()
            opt.step()
        if checkpoints and ep in checkpoints:
            seen[ep] = (accuracy(model, Xa, ya), accuracy(model, Xte, yte))
    return seen


def sweep(make, Xa, ya, epochs):
    """Train `make()` once per seed; return mean, std and parameter count."""
    scores = []
    for s in SEEDS:
        torch.manual_seed(s)
        m = make()
        fit(m, Xa, ya, epochs)
        scores.append(accuracy(m, Xte, yte))
    torch.manual_seed(0)
    return float(np.mean(scores)), float(np.std(scores)), n_params(make())

## 2.1 First, how long is long enough?

Every other measurement in this notebook depends on an epoch count, so that number has to
be settled **before** anything else — otherwise every curve below is a curve about a
budget rather than about the knob it claims to be about.

This is the rule stated in `planning/principles.md`, and it is worth saying plainly:
**never assume an epoch count for a number you are going to report.** Train to a plateau,
or show the curve. Not because it is tidy, but because it changes conclusions — twice in
this course's own history it has flipped the *sign* of a result.

In [ ]:
CHECKPOINTS = param([1, 2, 5, 10, 20, 40, 80], test=[1])
curves = {c: {"train": [], "test": []} for c in CHECKPOINTS}

for s in SEEDS:
    torch.manual_seed(s)
    model = NeuralNet(DIM, N_CLASSES)
    got = fit(model, Xtr, ytr, epochs=max(CHECKPOINTS), checkpoints=set(CHECKPOINTS))
    for c, (tr, te) in got.items():
        curves[c]["train"].append(tr)
        curves[c]["test"].append(te)

print(f"{'epoch':>6s} {'train':>8s} {'test':>18s}")
for c in CHECKPOINTS:
    tr, te = np.array(curves[c]["train"]), np.array(curves[c]["test"])
    print(f"{c:6d} {tr.mean():8.4f} {te.mean():12.4f} +- {te.std():.4f}")

In [ ]:
ep = np.array(CHECKPOINTS)
tr = np.array([np.mean(curves[c]["train"]) for c in CHECKPOINTS])
te = np.array([np.mean(curves[c]["test"]) for c in CHECKPOINTS])
sd = np.array([np.std(curves[c]["test"]) for c in CHECKPOINTS])

plt.figure(figsize=(7, 4))
plt.plot(ep, tr, "o-", label="train")
plt.errorbar(ep, te, yerr=sd, fmt="s-", capsize=3, label="test")
plt.xscale("log"); plt.xlabel("epochs (log)"); plt.ylabel("accuracy")
plt.title("compute: where does it stop paying?"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

Two things, and the second is the one people miss.

**Test accuracy plateaus almost immediately** — by about epoch 5 — and then does nothing
for the next 75. Training sixteen times longer buys you nothing at all; if anything it
drifts very slightly down.

**Train accuracy reaches 1.0000 and stays there.** The head memorises the training set
completely, and the test score does not care. That is worth sitting with: this is
textbook overfitting by the usual definition, and it costs nothing here, because the
ceiling is set by the *representation* rather than by the head's capacity to memorise.
A head cannot overfit its way past information the encoder never provided.

So: **20 epochs** for everything below. Comfortably past the plateau, cheap, and now a
justified number rather than a round one.

> **Compare with `5_unsupervised/01_why_pretext_tasks_work` §1.3, which reached the
> opposite conclusion.** There, stopping at 12 epochs hid a real effect that only appeared
> by epoch 60. Here, stopping at 5 would have lost nothing. Both were measured; neither is
> a default. *That* is why the rule is "check", not "train longer".

In [ ]:
EPOCHS = 20   # justified by the table above, not chosen

## 2.2 Axis one: parameters

The knob everyone reaches for first. Hold data and epochs fixed, and grow the head from a
bare linear map to something 140 times larger.

In [ ]:
heads = [
    ("linear only",      lambda: nn.Linear(DIM, N_CLASSES)),
    ("baseline 2-layer", lambda: NeuralNet(DIM, N_CLASSES)),
    ("hidden 64",        lambda: NeuralNet([DIM, 64], N_CLASSES)),
    ("hidden 256",       lambda: NeuralNet([DIM, 256], N_CLASSES)),
    ("hidden 1024",      lambda: NeuralNet([DIM, 1024], N_CLASSES)),
    ("hidden 256 x2",    lambda: NeuralNet([DIM, 256, 256], N_CLASSES)),
    ("hidden 1024 x3",   lambda: NeuralNet([DIM, 1024, 1024, 1024], N_CLASSES)),
]

param_curve = []
for name, mk in heads:
    m, sd_, p = sweep(mk, Xtr, ytr, EPOCHS)
    param_curve.append((name, p, m, sd_))
    print(f"{name:18s} {p:9,d} params   {m:.4f} +- {sd_:.4f}")

In [ ]:
ps = np.array([p for _, p, _, _ in param_curve])
ms = np.array([m for _, _, m, _ in param_curve])
ss = np.array([s for _, _, _, s in param_curve])

plt.figure(figsize=(7, 4))
plt.errorbar(ps, ms, yerr=ss, fmt="o", capsize=3)
for name, p, m, _ in param_curve:
    plt.annotate(name, (p, m), fontsize=7, xytext=(4, 4), textcoords="offset points")
plt.xscale("log"); plt.xlabel("parameters (log)"); plt.ylabel("test accuracy")
plt.title("parameters: 140x the capacity, and then?"); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

**About two points, across a 140-fold increase in parameters — and not monotone.** Look
carefully before calling it flat, because there is a real effect in here and it is
instructive that it is so small.

Widening the head *does* buy something: from the bare linear map to `hidden 1024` is
roughly +1.5 points, which is several times the seed noise, so it is a genuine gain rather
than a wobble. And then **depth hands it straight back.** `hidden 256 x2` is the worst
model on the list, and the 2.5-million-parameter `hidden 1024 x3` is second worst — both
below the plain two-layer baseline that has a fraction of their parameters.

So the honest description is not "flat" but **bounded and non-monotone**: there are about
one and a half points available on this axis, they are collected early, and spending more
after that actively costs you. Keep that number in mind — one and a half points — because
the next axis is about to move by thirty.

This is the single most important negative result in the course, and notebook 01 already
showed you the mechanism at one point in the space: capacity added *after* a frozen
encoder has almost nothing left to do. The encoder already did the work. Everything after
it is reading off an answer that is already there.

Note what this does to the phrase "bigger model". On this axis, bigger is not better and
it is not worse — it is **irrelevant**, which is a third possibility people rarely plan
for.

## 2.3 Axis two: data

Same head, same epochs, less data. Twenty-fold, from a few examples per class up to the
full training split.

In [ ]:
FRACTIONS = param([0.05, 0.1, 0.25, 0.5, 1.0], test=[1.0])
data_curve = []
for frac in FRACTIONS:
    n = int(frac * len(Xtr))
    m, sd_, _ = sweep(lambda: NeuralNet(DIM, N_CLASSES), Xtr[:n], ytr[:n], EPOCHS)
    data_curve.append((n, m, sd_))
    print(f"rows {n:5d} ({frac:>5.0%})   {m:.4f} +- {sd_:.4f}   "
          f"({n / N_CLASSES:.1f} per class)")

In [ ]:
ns = np.array([n for n, _, _ in data_curve])
dm = np.array([m for _, m, _ in data_curve])
dsd = np.array([s for _, _, s in data_curve])

plt.figure(figsize=(7, 4))
plt.errorbar(ns, dm, yerr=dsd, fmt="o-", capsize=3)
plt.xscale("log"); plt.xlabel("training rows (log)"); plt.ylabel("test accuracy")
plt.title("data: the one axis that moves"); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

**This one is real.** Thirty points of accuracy between the smallest and largest training
set — an order of magnitude more movement than the entire parameter sweep produced.

And it has the shape the scaling literature describes: roughly linear against the *log* of
the data, then flattening. The first doubling is worth far more than the last. From 5% to
10% of the data buys you more than from 50% to 100% does, which is the practical content
of a power law and the reason "collect more data" stops being good advice at some point.

Two consequences worth stating:

- **Where you are on this curve determines what advice is right.** At 141 rows the answer
  to "how do I improve this model" is *get more data*, and no architecture will save you.
  At 2820 rows the same advice buys a couple of points.
- **The steep part is where a frozen encoder earns its keep.** At 3 examples per class the
  model is still near 0.48 rather than near chance (1/47 ≈ 0.02), and that is entirely the
  encoder's doing.

## 2.4 The three axes together

Put them side by side, normalised so the comparison is about *movement* rather than about
absolute score.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)

axes[0].errorbar(ps, ms, yerr=ss, fmt="o", capsize=3)
axes[0].set_xscale("log"); axes[0].set_title(f"parameters  (span {np.ptp(ms) * 100:.1f} pt)")
axes[0].set_xlabel("parameters"); axes[0].set_ylabel("test accuracy")

axes[1].errorbar(ns, dm, yerr=dsd, fmt="o-", capsize=3, color="tab:green")
axes[1].set_xscale("log"); axes[1].set_title(f"data  (span {np.ptp(dm) * 100:.1f} pt)")
axes[1].set_xlabel("training rows")

axes[2].errorbar(ep, te, yerr=sd, fmt="s-", capsize=3, color="tab:orange")
axes[2].set_xscale("log"); axes[2].set_title(f"compute  (span {np.ptp(te) * 100:.1f} pt)")
axes[2].set_xlabel("epochs")

for a in axes:
    a.grid(alpha=.3)
plt.suptitle("three knobs, one dataset, one frozen encoder")
plt.tight_layout(); plt.show()

One of these is not like the others. **Data moves the metric by an order of magnitude more
than parameters or compute**, both of which are essentially flat over ranges spanning two
orders of magnitude.

That is a scaling law: not a formula, but a statement about *which resource is binding*.
And it is measured on your data rather than assumed from a paper, which matters, because —

## 2.5 The papers, and why the second one exists

Kaplan et al. (2020) — the PDF in `references/` — established that language-model loss
follows smooth power laws in parameters, data and compute, and concluded that when you get
more compute you should spend most of it on **more parameters**. That conclusion shaped
how models were built for two years.

Hoffmann et al. (2022), the *Chinchilla* paper, re-ran the experiment with a better
design — varying data and parameters *together* rather than holding one nearly fixed — and
found a substantially different optimum: models had been far too large for the data they
were trained on. A 70B model trained on more data beat a 280B model. The law was not
wrong; **the experiment behind it had a confound**, and finding it took no new idea, only
a more careful measurement.

That is this course's own standard, applied at a scale of millions of dollars. And it
rhymes with what you just measured: the axis that turned out to be underweighted in the
literature — data — is the only one that moved anything here.

Do not over-read the resemblance. Those are trillion-token language models and this is a
frozen texture encoder with a 47-class head; the mechanisms are different. What transfers
is the method, not the number.

## 2.6 The axis that is not capacity at all

Three knobs, and the honest summary of two of them is "does nothing". So what *does* move
this metric?

Swap the encoder. Same head, same data, same epochs — a different frozen representation
underneath.

In [ ]:
ENCODERS = param(
    ["dinov2-small", "resnet-18", "mobilenet_v2_1.0_224"],
    test=["dinov2-small"],
)

print(f"{'encoder':26s} {'dim':>5s} {'head params':>12s}   accuracy")
enc_rows = []
for enc in ENCODERS:
    c = VectorCache.from_hub(f"pttrn-io/dtd-{enc}")
    de = c.dataset.shuffle(seed=0).with_format("numpy")
    Xe = torch.tensor(np.asarray(de["embed"], dtype="float32"))
    ye = torch.tensor(np.asarray(de["label"]))
    ne = int(0.75 * len(Xe))
    Xte_, yte_ = Xe[ne:], ye[ne:]

    dim_e = Xe.shape[1]
    scores = []
    for s in SEEDS:
        torch.manual_seed(s)
        m = NeuralNet(dim_e, N_CLASSES)
        fit(m, Xe[:ne], ye[:ne], EPOCHS)
        scores.append(accuracy(m, Xte_, yte_))
    p = n_params(NeuralNet(dim_e, N_CLASSES))
    enc_rows.append((enc, dim_e, p, float(np.mean(scores)), float(np.std(scores))))
    print(f"{enc:26s} {dim_e:5d} {p:12,d}   {np.mean(scores):.4f} +- {np.std(scores):.4f}")

**Fifteen and a half points**, from a change that adds no parameters you control and takes
no extra training time.

And look at the parameter column while you do it. The weakest encoder here carries the
*largest* head — 1280 dimensions in, so roughly ten times the parameters of the winner's
head — and it loses by fifteen points. If parameters were the resource that mattered, that
row would not exist.

So the closing answer to the notebook's title question:

| knob | range swept | accuracy span |
|---|---|---|
| parameters | 140x | ~2 pt total; ~1.5 pt from width, then depth takes it back |
| compute | 80x | ~1 pt, flat past epoch 5 |
| data | 20x | **~30 pt**, saturating |
| **the encoder** | 3 choices | **~15 pt**, free |

**Capacity is not the resource. Information is.** Data adds information. A better encoder
adds information. Parameters and epochs only let you *use* information you already have —
and on a frozen representation you ran out of unused information almost immediately.

This is why lesson 1's two-line baseline kept winning, stated as a curve instead of as an
anecdote. And it is the question to carry into notebook 03: if the axes that matter are
not the ones you were going to search over, **what should a search actually search?**

## Exercises

**E1 — find the crossover.** §2.3 says at 141 rows the right advice is "get more data" and
at 2820 rows it is not. Find where that changes: fit a line to accuracy against log(rows)
over the first three points and over the last three, and report where the slope halves.
That number, not a rule of thumb, is when to stop collecting.

**E2 — does the flat parameter axis survive less data?** Repeat §2.2 at 10% of the
training rows. Capacity is supposed to matter more when data is scarce *or* less — predict
which before you run it, then explain the result you get. (Careful: with 282 rows, three
seeds may not be enough to separate anything. Check the error bars before believing the
ranking.)

**E3 — the encoder axis on a different dataset.** `evidence.md` §2 reports the encoder
spread as 14.4 points on DTD but only 2.4 on EuroSAT. Reproduce both. What property of a
dataset makes the encoder choice matter, and how would you tell in advance?

**E4 — a real scaling law.** Fit `accuracy = a - b * rows^(-c)` to §2.3's curve and
extrapolate: how many rows for 0.85? For 0.90? Then argue about whether you believe the
extrapolation, using the shape of the residuals rather than your intuition.

**E5 — the honest control on compute.** §2.1 concluded 20 epochs is enough *for this head
on this cache*. Re-run the epoch curve for the largest head from §2.2. Does a model with
2.5M parameters need longer to converge than one with 166k? If it does, does that change
the ranking in §2.2 — and does that mean the sweep was unfair?